# logit-thermo 探索：文章の DSC ——温度をゆっくり下げて上げ、凍る温度と溶ける温度のずれ（ヒステリシス）を測る下見（OLMo-2-1B、GPU、L4）

- **必要な secret：なし**（`HF_TOKEN` があれば取り込みが少し速くなるだけ）。**ランタイム：L4**（ハイメモリはいりません）。見込みは 1〜1.5 時間。終わったら自動で切れます。
- もう 1 本（Pythia-1B）のノートブックと、別のランタイムで同時に回せます（保存するファイルの名前が違うので、ぶつかりません）。
- 探索（下見、予想なし）。1 本の生成の中で温度を 0.85 → 0.35 → 0.85（と、その逆の順）にゆっくり動かし、写しの候補への場 h（−0.3・0・＋0.3）ごとに、凍る温度と溶ける温度のずれを測る材料を集めます。h ＝ 0 だけ、2 倍の速さの往復も。記事 40 × サンプル 4。
- 計算のコード：A2-2・A2-4・A2-7・A2-7b のデータ集めのコードと `scripts/a2_explore_field.py`（`0ac42c7ab82b06d2…`）は Drive の前のノートブックから取り出してハッシュで確かめる。`scripts/a2_explore_dsc.py`（`33f47a0a3bbda183…`）は Pythia-1B のノートブック（同じ Drive のフォルダ）から取り出してハッシュで確かめる。
- 保存先 `A2_explore/dsc/`。腕は 8 本。途中で切れても、もう一度「すべて実行」すれば終わった腕は飛ばします。**結果は表示しません。**
- **通知**：🚀・腕ごとの ✅（8 回）・🎉・⚠️。名札は「logit-thermo 探索 文章の DSC（ヒステリシスの下見、OLMo-2-1B、温度 0.85↔0.35 の往復 × 場 3 本、L4）」。終わったら、またはエラーで止まったら、ランタイムを切ります。


In [ ]:
# セル1：通知、GPU、Drive
import os, time, urllib.request
NTFY_TOPIC = 'YOUR_NTFY_TOPIC'
NB_LABEL = 'logit-thermo 探索 文章の DSC（ヒステリシスの下見、OLMo-2-1B、温度 0.85↔0.35 の往復 × 場 3 本、L4）'
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=(msg + '\n[' + NB_LABEL + ']').encode('utf-8'),
                                     headers={'Title': 'logit-thermo - explore DSC hysteresis OLMo-2-1B', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        import traceback
        e = result.error_in_exec
        txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
        v = os.environ.get('HF_TOKEN')
        if v: txt = txt.replace(v, '***')
        txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
        notify('⚠️ 止まりました（エラー）：\n' + txt, tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import torch
if not torch.cuda.is_available() or torch.cuda.get_device_properties(0).total_memory < 20e9:
    name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU なし'
    notify(f'⚠️ GPU のメモリが足りません（{name}）。L4 を選んでください。ランタイムを切ります', tags='warning')
    from google.colab import runtime; runtime.unassign()
    raise SystemExit
torch.backends.cuda.matmul.allow_tf32 = False; torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision('highest')
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# セル2：計算のコード
import ast, glob, hashlib, json
def take_src(want):
    """want：{変数の名前: sha256}。Drive の logit-thermo のノートブックから、埋め込んだコードを取り出してハッシュで確かめる。"""
    got = {}
    for p in sorted(glob.glob('/content/drive/MyDrive/logit-thermo' + '/**/*.ipynb', recursive=True)):
        try:
            cells = json.load(open(p))['cells']
        except Exception:
            continue
        for c in cells:
            s = c['source'] if isinstance(c['source'], str) else ''.join(c['source'])
            if c.get('cell_type') != 'code' or 'SRC' not in s:
                continue
            try:
                body = ast.parse(s).body
            except SyntaxError:
                continue
            for n in body:
                if isinstance(n, ast.Assign) and isinstance(n.targets[0], ast.Name) and n.targets[0].id in want and n.targets[0].id not in got:
                    v = ast.literal_eval(n.value)
                    if hashlib.sha256(v.encode()).hexdigest() == want[n.targets[0].id]:
                        got[n.targets[0].id] = v
        if len(got) == len(want):
            break
    miss = [k for k in want if k not in got]
    assert not miss, ('Drive のノートブックからコードを取り出せませんでした（前のノートブックを動かしていませんか）：', miss)
    return got
_s = take_src({'SRC22': 'e578cfc480980be6b0d1e18377b6beb8b7555e0744f56b13f63cb304a9f73d53', 'SRC24': '6ef87866ae7d895d63c6e07d48a58a2ddf4111a4802ffe98dbe7f61892c96b57', 'SRC7C': '8f8776cbe4a0e1a7b1f676c4f84ba70fb9c6c0b46ef1466969e09f0ba062b550', 'SRC7BC': 'b03ff9e2b16e2297fb9142a26e15f561caacf77216453d78e2024066c01c57d0', 'SRCF': '0ac42c7ab82b06d2e7443ba9c0c0e147c0121b5351e0b5830b1a54b46043190d', 'SRCD': '33f47a0a3bbda18341205dedb94c6d200f295fc4136f5cb8e0ee067b2d7f518c'})
SRC22, SRC24, SRC7C, SRC7BC, SRCF = _s['SRC22'], _s['SRC24'], _s['SRC7C'], _s['SRC7BC'], _s['SRCF']
SRCF, SRCD = _s['SRCF'], _s['SRCD']   # 名前を 2 つ並べて置く（take_src が読み飛ばす形）
exec(SRC22)
exec(SRC24)
exec(SRC7C)
exec(SRC7BC)
exec(SRCF)
exec(SRCD)
print('コードを置きました')


In [ ]:
# セル3：自己テスト → 記事（A2-2 と同じか確かめる）→ 8 本の腕 → 保存（結果は表示しない）
import json, sys, transformers
try:
    print('自己テスト：', self_test_a27('cuda'), self_test_7b('cuda'), self_test_field('cuda'), self_test_dsc('cuda'))
    os.makedirs('/content/drive/MyDrive/logit-thermo/A2_explore/dsc', exist_ok=True)
    assert SEED == 20261027, ('記事を選ぶ種が A2-2 と違います', SEED)
    docs, meta = load_docs()
    old = json.load(open('/content/drive/MyDrive/logit-thermo' + '/A2_2/docs_meta.json'))
    assert [d['text_sha256'] for d in old['docs']] == [d['text_sha256'] for d in meta['docs']], 'BBC の記事が A2-2 と違います'
    name, hf_id = D_MODELS['olmo']
    run_dsc(name, hf_id, docs, '/content/drive/MyDrive/logit-thermo/A2_explore/dsc', 'cuda', notify=lambda s: notify('✅ ' + s, tags='white_check_mark'))
    json.dump(dict(model=[name, hf_id], arms=D_ARMS, legs=D_LEG, T=[D_TH, D_TL], samples=D_S, chunk=D_CHUNK, seed=D_SEED,
                   field_sha256='0ac42c7ab82b06d2e7443ba9c0c0e147c0121b5351e0b5830b1a54b46043190d', dsc_sha256='33f47a0a3bbda18341205dedb94c6d200f295fc4136f5cb8e0ee067b2d7f518c', torch=torch.__version__, transformers=transformers.__version__,
                   python=sys.version.split()[0], gpu=torch.cuda.get_device_name(0), matmul=torch.get_float32_matmul_precision()),
              open('/content/drive/MyDrive/logit-thermo/A2_explore/dsc' + f'/run_meta_{name}.json', 'w'), ensure_ascii=False, indent=1)
    print('保存しました')
except Exception as e:
    import traceback
    _NOTIFIED['error'] = True
    txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
    v = os.environ.get('HF_TOKEN')
    if v: txt = txt.replace(v, '***')
    txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
    notify('⚠️ 探索 文章の DSC（OLMo-2-1B）で止まりました。ランタイムを切ります：\n' + txt, tags='warning')
    if globals().get('AUTO_DISCONNECT', True):
        from google.colab import drive as _d, runtime as _r
        _d.flush_and_unmount(); time.sleep(3); _r.unassign()
    raise


In [ ]:
# 最後：終わりの通知と、ランタイムを切る
notify('🎉 終わりました：文章の DSC（OLMo-2-1B）の材料を A2_explore/dsc に保存しました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount(); time.sleep(3); _runtime.unassign()
